In [3]:
import os
import requests
import pandas as pd
import datetime
import time
import re
from dotenv import load_dotenv

load_dotenv("../../config/.env") 
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

# --- 1. Configuration ---
SOURCE_PREFIX = "LCDGT"
SOURCE_NAME = "LC Demographic Group Terms"
BASE_URI = "http://id.loc.gov/authorities/demographicTerms/"

notebook_dir = os.path.dirname(os.path.abspath("__file__"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))

# FILE NAMING
raw_ingest_file = os.path.join(raw_data_dir, "lcdgt_religion_raw.csv")
master_file = os.path.join(raw_data_dir, "metadata_ingest_undeduped.csv")
prefix_map_file = os.path.join(raw_data_dir, "prefix_map.csv")

COLLECTION_URL = "https://id.loc.gov/search/?q=memberOf:http://id.loc.gov/authorities/demographicTerms/collection_LCDGT_Religion&fo=json&count=500"

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})',
    'Accept': 'application/json'
}

COLUMN_ORDER = [
    "Source_System", "Primary_Label", "CURIE", "Formal_Label", 
    "Hierarchy_Path", "Synonyms", "Description", "Parent_IDs", 
    "Concept_ID", "URI", "Status", "Extraction_Date"
]

os.makedirs(raw_data_dir, exist_ok=True)

# --- 2. Prefix Map Update ---
new_row = {"Prefix": SOURCE_PREFIX, "Base_URI": BASE_URI, "Source_Name": SOURCE_NAME}

if os.path.exists(prefix_map_file):
    df_prefixes = pd.read_csv(prefix_map_file)
    if SOURCE_PREFIX not in df_prefixes['Prefix'].values:
        df_prefixes = pd.concat([df_prefixes, pd.DataFrame([new_row])], ignore_index=True)
        df_prefixes.to_csv(prefix_map_file, index=False, encoding='utf-8-sig')
        print(f"Added {SOURCE_PREFIX} to prefix_map.csv")
else:
    df_prefixes = pd.DataFrame([new_row])
    df_prefixes.to_csv(prefix_map_file, index=False, encoding='utf-8-sig')
    print(f"Created prefix_map.csv and registered {SOURCE_PREFIX}.")

print(f"LCDGT Ready.\nRaw: {raw_ingest_file}\nMaster: {master_file}")

Created prefix_map.csv and registered LCDGT.
LCDGT Ready.
Raw: c:\Users\njsgi\Documents\GitHub\religion-ontology-mapping\data\raw\lcdgt_religion_raw.csv
Master: c:\Users\njsgi\Documents\GitHub\religion-ontology-mapping\data\raw\metadata_ingest_undeduped.csv


In [2]:
label_cache = {}
member_uris = set()

def get_label_only(uri):
    if not uri: return ""
    clean_u = uri.rstrip('.json').rstrip('.rdf').rstrip('/')
    if clean_u in label_cache: return label_cache[clean_u]
    try:
        res = requests.get(f"{clean_u}.json", headers=HEADERS, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if isinstance(data, dict): data = [data]
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_u), {})
            label = node.get('http://www.w3.org/2004/02/skos/core#prefLabel', [{}])[0].get('@value', '')
            if label:
                label_cache[clean_u] = label
                return label
    except: pass
    return clean_u.split('/')[-1]

def get_lc_details(uri):
    clean_uri = uri.rstrip('/').replace('.json', '').replace('.rdf', '').replace('.html', '')
    # The crucial fix: retry up to 3 times to survive LC server drops
    for attempt in range(3):
        try:
            res = requests.get(f"{clean_uri}.json", headers=HEADERS, timeout=15)
            if res.status_code != 200: 
                time.sleep(1)
                continue
            
            data = res.json()
            if isinstance(data, dict): data = [data]
            node = next((item for item in data if isinstance(item, dict) and item.get('@id') == clean_uri), {})
            
            SKOS_PREF = 'http://www.w3.org/2004/02/skos/core#prefLabel'
            SKOS_ALT  = 'http://www.w3.org/2004/02/skos/core#altLabel'
            SKOS_BROADER = 'http://www.w3.org/2004/02/skos/core#broader'
            MADS_CITATION = 'http://www.loc.gov/mads/rdf/v1#citationNote'

            label = node.get(SKOS_PREF, [{}])[0].get('@value', 'No Label')
            label_cache[clean_uri] = label
            
            synonyms = [s.get('@value') for s in node.get(SKOS_ALT, []) if s.get('@value')]
            
            descriptions = []
            for item in data:
                if isinstance(item, dict):
                    notes = item.get(MADS_CITATION, [])
                    for n in notes:
                        if n.get('@value'): descriptions.append(n.get('@value'))
            
            broader_nodes = node.get(SKOS_BROADER, [])
            p_ids = [b.get('@id').split('/')[-1] for b in broader_nodes if b.get('@id')]
            p_labels = [get_label_only(b.get('@id')) for b in broader_nodes if b.get('@id')]

            return {
                'label': label,
                'synonyms': " | ".join(list(set(synonyms))),
                'description': " | ".join(descriptions),
                'parent_ids': " | ".join(p_ids),
                'parent_labels': " | ".join(p_labels)
            }
        except Exception:
            time.sleep(2) # Back off and try again
    return None

# --- 1. Discovery ---
print("Discovering Collection...")

try:
    response = requests.get(COLLECTION_URL, headers=HEADERS, timeout=20)
    if response.status_code == 200:
        search_data = response.json()
        
        def extract_uris(obj):
            if isinstance(obj, dict):
                for k, v in obj.items():
                    if isinstance(v, str) and '/demographicTerms/dg' in v:
                        match = re.search(r'(dg\d+)', v)
                        if match: 
                            member_uris.add(f"{BASE_URI}{match.group(1)}")
                    extract_uris(v)
            elif isinstance(obj, list):
                for i in obj: 
                    extract_uris(i)

        extract_uris(search_data)
        print(f"Discovery complete. Found {len(member_uris)} URIs.")
    else:
        print(f"Search failed. HTTP Status: {response.status_code}")
except Exception as e:
    print(f"Discovery error: {e}")

# --- 2. Ingest ---
total = len(member_uris)
if total > 0:
    print(f"\nStarting ingest for {total} terms...")
    if os.path.exists(raw_ingest_file): os.remove(raw_ingest_file)
    
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    for i, uri in enumerate(sorted(list(member_uris)), 1):
        cid = uri.split('/')[-1]
        
        meta = get_lc_details(uri)
        
        if meta:
            path = f"{meta['parent_labels']} > {meta['label']}" if meta['parent_labels'] else meta['label']
            
            display_text = f"[{i}/{total}] Ingesting: {meta['label'][:60]}"
            print(f"\r{display_text}{' ' * 40}", end="", flush=True)

            row = {
                'Source_System': SOURCE_NAME, 'Primary_Label': meta['label'],
                'CURIE': f"{SOURCE_PREFIX}:{cid}", 'Formal_Label': meta['label'],
                'Hierarchy_Path': path, 'Synonyms': meta['synonyms'],
                'Description': meta['description'], 'Parent_IDs': meta['parent_ids'],
                'Concept_ID': cid, 'URI': uri, 'Status': 'active', 'Extraction_Date': timestamp
            }
            pd.DataFrame([row])[COLUMN_ORDER].to_csv(raw_ingest_file, mode='a', index=False, 
                                                  header=not os.path.exists(raw_ingest_file), encoding='utf-8-sig')
        time.sleep(1.0) 
else:
    print("\nNo URIs found to ingest.")

print(f"\n\nIngest Complete. Total saved: {len(pd.read_csv(raw_ingest_file)) if os.path.exists(raw_ingest_file) else 0}")

Discovering Collection...
Discovery complete. Found 95 URIs.

Starting ingest for 95 terms...
[17/95] Ingesting: Lutherans                                                  

KeyboardInterrupt: 

In [7]:
if os.path.exists(raw_ingest_file):
    df_new = pd.read_csv(raw_ingest_file)
    
    if os.path.exists(master_file):
        df_master = pd.read_csv(master_file)
        
        # Deduplication Check
        existing_uris = set(df_master['URI'].dropna().unique())
        df_to_add = df_new[~df_new['URI'].isin(existing_uris)]
        
        if not df_to_add.empty:
            df_final = pd.concat([df_master, df_to_add], ignore_index=True)
            df_final.to_csv(master_file, index=False, encoding='utf-8-sig')
            print(f"Added {len(df_to_add)} LCDGT records to master.")
        else:
            print("No new LCDGT records found to add.")
    else:
        df_new.to_csv(master_file, index=False, encoding='utf-8-sig')
        print(f"Master file created with {len(df_new)} LCDGT records.")
else:
    print(f"Error: {raw_ingest_file} not found. Run Cell 2 first.")

Added 95 LCDGT records to master.
